## load packages

In [1]:
import pandas as pd
import numpy as np

In [2]:
import os

print(os.getcwd())

/home/jovyan/sp25-student/grad-proj/nlp-chatbot-analysis


In [3]:
import warnings
warnings.filterwarnings('ignore')

## load data

In [4]:
# Main Dataset

# Conversation Data -- we will use this data in the "Conversation Data" section
df = pd.read_json(
    "/home/jovyan/_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/training-set/chatbot-arena-conversations.jsonl.gz",
    lines=True,
    compression="gzip"
)

print(df.shape)
df.head(5)

(25282, 7)


,question_id,model_a,model_b,winner,judge,conversation_a,conversation_b
0,58210e39b3fd4441a2bd4a518bb44c2d,chatglm-6b,koala-13b,model_b,arena_user_973,[{'content': 'What is the difference between O...,[{'content': 'What is the difference between O...
1,2564acd09e3942fd97657d05282d4389,oasst-pythia-12b,alpaca-13b,tie,arena_user_973,[{'content': 'Why did my parent not invite me ...,[{'content': 'Why did my parent not invite me ...
2,90bfd142157948aba01931726c888e7f,koala-13b,oasst-pythia-12b,model_b,arena_user_973,"[{'content': 'Fuji vs. Nikon, which is better?...","[{'content': 'Fuji vs. Nikon, which is better?..."
3,a7c5accc53e649a3bc6b2e41d962ebc4,vicuna-13b,oasst-pythia-12b,model_b,arena_user_973,[{'content': 'How to build an arena for chatbo...,[{'content': 'How to build an arena for chatbo...
4,adf27e819a3c494cb6e993f0c660e097,vicuna-13b,koala-13b,model_a,arena_user_973,"[{'content': 'When is it today?', 'role': 'use...","[{'content': 'When is it today?', 'role': 'use..."


In [5]:
# Auxiliary Datasets

# Embedding Data -- we will use this data in the "Embedding Data" section
prompt_embeddings = np.load(
    "/home/jovyan/_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/training-set/chatbot-arena-prompts-embeddings.npy"
)

response_a_embeddings = np.load(
    "/home/jovyan/_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/training-set/chatbot-arena-model_a_response-embeddings.npy"
)

response_b_embeddings = np.load(
    "/home/jovyan/_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/training-set/chatbot-arena-model_b_response-embeddings.npy"
)

# Topic Modeling and Hardness Score Data -- we will use this data in the "Topic Modeling and Hardness Score Data" section
topic_and_hardness = pd.read_json(
    "/home/jovyan/_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/training-set/chatbot-arena-gpt3-scores.jsonl.gz",
    lines=True,
    compression="gzip"
)

In [6]:
print(prompt_embeddings.shape)
prompt_embeddings

(25282, 256)


array([[-0.12376316, -0.1173524 ,  0.04567662, ..., -0.02439648,
        -0.03724024, -0.04380682],
       [ 0.00602781,  0.02843601, -0.09102212, ...,  0.08506154,
        -0.05333152,  0.00185428],
       [-0.03522179, -0.10940242, -0.02224718, ..., -0.14116742,
         0.00477351,  0.00416916],
       ...,
       [ 0.02476096, -0.02741823,  0.07049022, ..., -0.05541623,
        -0.02461602,  0.06005438],
       [ 0.01620374,  0.04473886,  0.08496623, ..., -0.02189322,
        -0.05438842,  0.02849752],
       [-0.04179214,  0.00963908, -0.00338678, ...,  0.08221702,
        -0.02381286, -0.10591594]], dtype=float32)

In [7]:
print(topic_and_hardness.shape)
topic_and_hardness.head(5)

(25282, 12)


,question_id,prompt,openai_scores_raw_choices_nested,topic_modeling_1,score_reason_1,score_value_1,topic_modeling_2,score_reason_2,score_value_2,topic_modeling_3,score_reason_3,score_value_3
0,58210e39b3fd4441a2bd4a518bb44c2d,What is the difference between OpenCL and CUDA?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Technical Comparison,This prompt requires the AI to accurately comp...,9,Software Comparison,This prompt assesses the AI's factual accuracy...,8,"Comparison, Technology",This prompt requires the AI to demonstrate kno...,9
1,2564acd09e3942fd97657d05282d4389,Why did my parent not invite me to their wedding?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...","Reasoning, Emotion",This prompt requires the AI to understand huma...,9,"Emotions, Relationships",This prompt involves understanding complex hum...,8,"Reasoning, Emotional",This prompt challenges the AI to infer motives...,8
2,90bfd142157948aba01931726c888e7f,"Fuji vs. Nikon, which is better?","[{'finish_reason': 'stop', 'index': 0, 'logpro...",Camera comparison,This prompt does not require problem-solving s...,2,Comparative Analysis,This prompt assesses the AI's ability to analy...,6,Photography comparison,This prompt is subjective and does not provide...,2
3,a7c5accc53e649a3bc6b2e41d962ebc4,How to build an arena for chatbots?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Chatbot Arena,This prompt requires problem-solving skills an...,8,Chatbot Arena,This prompt requires the AI to engage in probl...,8,Chatbot Arena,This prompt requires problem-solving skills an...,8
4,adf27e819a3c494cb6e993f0c660e097,When is it today?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Time Query,This prompt is very straightforward and does n...,2,Date Inquiry,This prompt is very straightforward and does n...,2,Time-based Inquiry,This prompt is too straightforward and simply ...,2


In [8]:
hayden_cluster = pd.read_csv('hayden_cluster_train.csv')

print(hayden_cluster.shape)
hayden_cluster.head(5)

(25282, 15)


,question_id,prompt,openai_scores_raw_choices_nested,topic_modeling_1,score_reason_1,score_value_1,topic_modeling_2,score_reason_2,score_value_2,topic_modeling_3,score_reason_3,score_value_3,topic_modeling_1_cluster,topic_modeling_2_cluster,topic_modeling_3_cluster
0,58210e39b3fd4441a2bd4a518bb44c2d,What is the difference between OpenCL and CUDA?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Technical Comparison,This prompt requires the AI to accurately comp...,9,Software Comparison,This prompt assesses the AI's factual accuracy...,8,"Comparison, Technology",This prompt requires the AI to demonstrate kno...,9,16,0,16
1,2564acd09e3942fd97657d05282d4389,Why did my parent not invite me to their wedding?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...","Reasoning, Emotion",This prompt requires the AI to understand huma...,9,"Emotions, Relationships",This prompt involves understanding complex hum...,8,"Reasoning, Emotional",This prompt challenges the AI to infer motives...,8,1,1,1
2,90bfd142157948aba01931726c888e7f,"Fuji vs. Nikon, which is better?","[{'finish_reason': 'stop', 'index': 0, 'logpro...",Camera comparison,This prompt does not require problem-solving s...,2,Comparative Analysis,This prompt assesses the AI's ability to analy...,6,Photography comparison,This prompt is subjective and does not provide...,2,2,16,2
3,a7c5accc53e649a3bc6b2e41d962ebc4,How to build an arena for chatbots?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Chatbot Arena,This prompt requires problem-solving skills an...,8,Chatbot Arena,This prompt requires the AI to engage in probl...,8,Chatbot Arena,This prompt requires problem-solving skills an...,8,19,19,19
4,adf27e819a3c494cb6e993f0c660e097,When is it today?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Time Query,This prompt is very straightforward and does n...,2,Date Inquiry,This prompt is very straightforward and does n...,2,Time-based Inquiry,This prompt is too straightforward and simply ...,2,16,16,16


## data understanding

In [9]:
# missing value exploration

In [10]:
df.isna().sum()

question_id       0
model_a           0
model_b           0
winner            0
judge             0
conversation_a    0
conversation_b    0
dtype: int64

In [11]:
hayden_cluster.isna().sum()

question_id                          0
prompt                               0
openai_scores_raw_choices_nested     0
topic_modeling_1                    26
score_reason_1                      26
score_value_1                       26
topic_modeling_2                    26
score_reason_2                      26
score_value_2                       26
topic_modeling_3                    26
score_reason_3                      26
score_value_3                       26
topic_modeling_1_cluster             0
topic_modeling_2_cluster             0
topic_modeling_3_cluster             0
dtype: int64

In [12]:
hayden_cluster[hayden_cluster['topic_modeling_1'].isna()]

,question_id,prompt,openai_scores_raw_choices_nested,topic_modeling_1,score_reason_1,score_value_1,topic_modeling_2,score_reason_2,score_value_2,topic_modeling_3,score_reason_3,score_value_3,topic_modeling_1_cluster,topic_modeling_2_cluster,topic_modeling_3_cluster
584,e6d45ead33114cca8ee3cfa028517eff,I want you to act as a linux terminal. I will ...,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5060,addaa796ee094f029f8014ea1468df8a,\nAssume the role of an API that provides a ch...,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5458,d37eb99864fa41ecab49026abdddb53e,I want you to act as a javascript console. I w...,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5595,6da02001e74041d0947982fb4d05db9e,I want you to act as a linux terminal. I will ...,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
6260,d93e36df73e84aa2ade15d4a038c098f,# User Input\n## This is what the user request...,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7807,50b63f92bc5948218e1555d1eae17797,"Submit a JSON object, exclusively, with the sp...","[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7808,8277b16d9a0845d694a33c04f446926c,"Submit a JSON object, exclusively, with the sp...","[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7809,d56d698d4c1c495682a366f2a78fcb77,"Submit a JSON object, exclusively, with the sp...","[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
8529,86c7abedb5f84b7ea752cc98d324d387,"Reply in the format: {""question"":""<question>"",...","[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
10755,974569d7b9c74ca591f1922bf3722266,"Introducing MPT-7B, the latest entry in our Mo...","[{'finish_reason': 'stop', 'index': 0, 'logpro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [13]:
# are there any duplicated training samples?

# use index to merge
# variability in the provided topic modeling

In [14]:
df[df.duplicated(subset=['question_id','model_a','model_b','winner'], keep=False)]

,question_id,model_a,model_b,winner,judge,conversation_a,conversation_b
12852,ece135ff5264475d8fa3f9e1198b652f,palm-2,claude-v1,tie,arena_user_7499,[{'content': 'what are the medical exams in th...,[{'content': 'what are the medical exams in th...
12856,ece135ff5264475d8fa3f9e1198b652f,palm-2,claude-v1,tie,arena_user_7499,[{'content': 'what are the medical exams in th...,[{'content': 'what are the medical exams in th...
15040,00b5a7d8b1c04eda8a350b4a899b1f8e,stablelm-tuned-alpha-7b,gpt-4,model_b,arena_user_13266,"[{'content': 'In python, how do I compare two ...","[{'content': 'In python, how do I compare two ..."
15042,00b5a7d8b1c04eda8a350b4a899b1f8e,stablelm-tuned-alpha-7b,gpt-4,model_b,arena_user_13266,"[{'content': 'In python, how do I compare two ...","[{'content': 'In python, how do I compare two ..."


In [15]:
hayden_cluster[hayden_cluster.duplicated(subset=['question_id'], keep=False)]

,question_id,prompt,openai_scores_raw_choices_nested,topic_modeling_1,score_reason_1,score_value_1,topic_modeling_2,score_reason_2,score_value_2,topic_modeling_3,score_reason_3,score_value_3,topic_modeling_1_cluster,topic_modeling_2_cluster,topic_modeling_3_cluster
1576,20cb94a0b7c147fe8fc6250b2a20fd1c,Write a song for my favourite singer: Zhou She...,"[{'finish_reason': 'stop', 'index': 0, 'logpro...","Songwriting, Celebration",This prompt requires the AI to demonstrate cre...,8,Lyric Composition,This prompt requires creativity to compose a s...,8,Lyric Songwriting,This prompt challenges the AI to be creative i...,8,3,3,3
1577,20cb94a0b7c147fe8fc6250b2a20fd1c,Write a song for my favourite singer: Zhou She...,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Lyric composition,This prompt requires the AI to create a song l...,8,Lyric Composition,This prompt requires the AI to create a song l...,8,Lyric Composition,This prompt requires creativity in composing a...,8,3,3,3
5129,c6192fdbd3dd46c98791d7abac92b56d,What is the best scheme implementation?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Scheme Implementation,This prompt requires the AI to showcase its pr...,8,Scheme Implementation,This prompt requires the AI to demonstrate its...,8,Scheme Implementation,This prompt requires the AI to demonstrate pro...,8,10,10,10
5130,c6192fdbd3dd46c98791d7abac92b56d,What is the best scheme implementation?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Scheme Implementation,This prompt is too broad and lacks specificity...,2,"Scheme, Implementation","This prompt is very vague and open-ended, lack...",2,Scheme Implementation,This prompt assesses the AI's problem-solving ...,7,10,10,10
6596,49c49843372a4510926e9a572dbcb6f7,"Act as reddit user, Act as a human, Act as 17 ...","[{'finish_reason': 'stop', 'index': 0, 'logpro...",Social Interaction,This prompt aims to assess the AI's ability to...,7,"Communication, Social Interaction",This prompt focuses on the AI's ability to sim...,7,Social Interaction,This prompt assesses the AI's ability to engag...,8,19,19,19
6597,49c49843372a4510926e9a572dbcb6f7,"Act as reddit user, Act as a human, Act as 17 ...","[{'finish_reason': 'stop', 'index': 0, 'logpro...","Communication, Direct Message",The prompt assesses the AI's ability to engage...,7,Social Interaction,This prompt assesses the AI's ability to engag...,7,"Role-playing, Direct Messaging",This prompt primarily assesses the AI's abilit...,6,19,19,28
9345,c3a1ad0511084ca68a264ad16fac8bdf,Hey there. How are you today?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...","Greeting, Social Interaction",This prompt does not assess problem-solving ab...,2,Emotional check-in,This prompt does not provide a clear task for ...,2,Emotional Check-in,"This prompt does not assess problem-solving, c...",2,19,1,1
9346,c3a1ad0511084ca68a264ad16fac8bdf,Hey there. How are you today?,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Greeting Response,This prompt is more of a casual greeting and d...,1,Social Interaction,This prompt does not evaluate problem-solving ...,1,Small Talk,This prompt is simple small talk and does not ...,2,19,19,19
10462,1dd6137eb3c3470989e18ab729ccc0b3,write short telugu poem,"[{'finish_reason': 'stop', 'index': 0, 'logpro...",Creative Writing,This prompt requires creativity and linguistic...,9,"Creativity, Poetry",This prompt requires the AI to demonstrate cre...,8,"Creativity, Poetry",The prompt challenges the AI to showcase creat...,8,12,3,3
10463,1dd6137eb3c3470989e18ab729ccc0b3,write short telugu poem,"[{'finish_reason': 'stop', 'index': 0, 'logpro...","Creativity, Poetry",Writing a poem in a different language require...,8,Poetry Creation,This prompt requires creativity and language p...,7,Poetry Generation,Generating a poem in a different language requ...,8,3,3,3


In [16]:
# number of different models

In [17]:
models = set((list(df['model_a']) + list(df['model_b'])))
print(len(models))
models

20


{'RWKV-4-Raven-14B',
 'alpaca-13b',
 'chatglm-6b',
 'claude-instant-v1',
 'claude-v1',
 'dolly-v2-12b',
 'fastchat-t5-3b',
 'gpt-3.5-turbo',
 'gpt-4',
 'gpt4all-13b-snoozy',
 'guanaco-33b',
 'koala-13b',
 'llama-13b',
 'mpt-7b-chat',
 'oasst-pythia-12b',
 'palm-2',
 'stablelm-tuned-alpha-7b',
 'vicuna-13b',
 'vicuna-7b',
 'wizardlm-13b'}

In [18]:
# number of unique model a/b competitions

In [19]:
(19+1)*19/2

190.0

In [20]:
# ground truth labels count

In [21]:
df['winner'].value_counts()

winner
model_a          9002
model_b          8862
tie (bothbad)    4632
tie              2786
Name: count, dtype: int64

In [22]:
# number of topics

In [23]:
hayden_cluster[['topic_modeling_1','topic_modeling_2','topic_modeling_3']].describe()

,topic_modeling_1,topic_modeling_2,topic_modeling_3
count,25256,25256,25256
unique,11289,11182,11175
top,Creative Writing,Creative Writing,Creative Writing
freq,565,586,596


## save complete data

In [24]:
# remove samples (with no hardness score or prompt topic lables) across all of our datasets

temp_indices_to_remove = hayden_cluster[hayden_cluster['topic_modeling_1'].isna()].index

prompt_embeddings = np.delete(prompt_embeddings, temp_indices_to_remove, axis=0)
response_a_embeddings = np.delete(response_a_embeddings, temp_indices_to_remove, axis=0)
response_b_embeddings = np.delete(response_b_embeddings, temp_indices_to_remove, axis=0)

np.save('prompt_embeddings', prompt_embeddings)
np.save('response_a_embeddings', response_a_embeddings)
np.save('response_b_embeddings', response_b_embeddings)

print(prompt_embeddings.shape)
print(response_a_embeddings.shape)
print(response_b_embeddings.shape)

hayden_cluster = hayden_cluster[hayden_cluster['topic_modeling_1'].notna()]

(25256, 256)
(25256, 256)
(25256, 256)


In [25]:
# clean the hardness score columns
temp_cols = ['score_value_1','score_value_2','score_value_3']
hayden_cluster[temp_cols] = hayden_cluster[temp_cols].replace(to_replace={'[[8]]':'8', 
                                                                          '[[7]]':'7',
                                                                          '[[2]]':'2',
                                                                          '[[3]]':'3',
                                                                          '[[4]]':'4',
                                                                          '[[6]]':'6',
                                                                          '[[9]]':'9',
                                                                          '0.8':'8', 
                                                                          '0.7000000000000001':'7'})

df_merged = pd.merge(left=hayden_cluster[['prompt',
                                          'topic_modeling_1','score_reason_1','score_value_1',
                                          'topic_modeling_2','score_reason_2','score_value_2',
                                          'topic_modeling_3','score_reason_3','score_value_3',
                                          'topic_modeling_1_cluster','topic_modeling_2_cluster','topic_modeling_3_cluster']], 
                     right=df[['model_a','model_b','winner']], 
                     how='inner', 
                     left_index=True, right_index=True)

df_merged.to_csv('complete_data_for_tasks_undeciphered.csv', index=False)

print(df_merged.shape)
df_merged.head(5)

(25256, 16)


,prompt,topic_modeling_1,score_reason_1,score_value_1,topic_modeling_2,score_reason_2,score_value_2,topic_modeling_3,score_reason_3,score_value_3,topic_modeling_1_cluster,topic_modeling_2_cluster,topic_modeling_3_cluster,model_a,model_b,winner
0,What is the difference between OpenCL and CUDA?,Technical Comparison,This prompt requires the AI to accurately comp...,9,Software Comparison,This prompt assesses the AI's factual accuracy...,8,"Comparison, Technology",This prompt requires the AI to demonstrate kno...,9,16,0,16,chatglm-6b,koala-13b,model_b
1,Why did my parent not invite me to their wedding?,"Reasoning, Emotion",This prompt requires the AI to understand huma...,9,"Emotions, Relationships",This prompt involves understanding complex hum...,8,"Reasoning, Emotional",This prompt challenges the AI to infer motives...,8,1,1,1,oasst-pythia-12b,alpaca-13b,tie
2,"Fuji vs. Nikon, which is better?",Camera comparison,This prompt does not require problem-solving s...,2,Comparative Analysis,This prompt assesses the AI's ability to analy...,6,Photography comparison,This prompt is subjective and does not provide...,2,2,16,2,koala-13b,oasst-pythia-12b,model_b
3,How to build an arena for chatbots?,Chatbot Arena,This prompt requires problem-solving skills an...,8,Chatbot Arena,This prompt requires the AI to engage in probl...,8,Chatbot Arena,This prompt requires problem-solving skills an...,8,19,19,19,vicuna-13b,oasst-pythia-12b,model_b
4,When is it today?,Time Query,This prompt is very straightforward and does n...,2,Date Inquiry,This prompt is very straightforward and does n...,2,Time-based Inquiry,This prompt is too straightforward and simply ...,2,16,16,16,vicuna-13b,koala-13b,model_a
